# MERT: Music Understanding Model — Colab Demo

**MERT** (*Acoustic Music undERstanding Model with Large-scale Self-supervised Training*, [arXiv:2306.00107](https://arxiv.org/abs/2306.00107), ICLR 2024) is a self-supervised pre-trained model for music audio — think of it as *HuBERT for music*.

This notebook:
1. Dissects the full MERT architecture **layer by layer** (convolutional front-end → feature projection → 12-layer Transformer encoder)
2. Traces tensor shapes through every stage of the model
3. Extracts the 13 hidden-state layers and shows what each depth level captures
4. Computes layer-wise cosine similarity between two tracks
5. Builds a self-similarity heatmap (music structure analysis)
6. Shows the standard pattern for hooking MERT into a downstream task
7. Plays the audio inline — always listen to your data before you model it
8. Runs a real-data experiment: **FMA genre classification with a layer-by-layer linear probe** (accuracy-vs-layer curve)

Every step comes as a **text explanation + figure** pair.

> Runtime tip: **Runtime → Change runtime type → T4 GPU** recommended. Everything also runs on CPU, just slower (the §12 feature pass noticeably so).

## 1. MERT Architecture Overview

MERT is a HuBERT-family encoder (`MERTModel` extends `HubertModel`, swapping in a custom `MERTFeatureProjection`). Audio goes in as a **24 kHz mono waveform** and comes out as **13 hidden states** (1 CNN front-end output + 12 Transformer layers), each of shape `(frames, 768)` at **75 frames/sec**:

```
waveform (24 kHz mono)
   │
   ▼
┌─────────────────────────────────────────────┐
│ Convolutional Feature Encoder               │  7 conv layers, 512 channels each
│ kernel (10,3,3,3,3,2,2)                     │  stride (5,2,2,2,2,2,2)  → ÷320 total
└─────────────────────────────────────────────┘
   │ (samples/320 × 512)  — 75 fps
   ▼
┌─────────────────────────────────────────────┐
│ Feature Projection                          │  LayerNorm + Linear 512 → 768
└─────────────────────────────────────────────┘
   │ (frames × 768)            ← hidden state 0
   ▼
┌─────────────────────────────────────────────┐
│ Transformer Encoder × 12                    │  each layer:
│                                             │    LN → Multi-Head Self-Attention (12 heads)
│                                             │    → LN → FFN (768 → 3072 → 768)
└─────────────────────────────────────────────┘
   │                           ← hidden states 1..12
   ▼
13 hidden states, each (frames × 768)
```

**How it was pre-trained (teacher–student pseudo-labeling).** No labels are needed: two complementary teachers produce pseudo-labels, and the student Transformer learns to predict them at masked positions:

| Teacher | Implementation | Captures |
|---|---|---|
| Acoustic teacher | RVQ-VAE / neural codec tokens (8 codebooks × 1024 entries, EnCodec-style) | timbre, spectral detail |
| Music teacher | masked CQT features (computed on-the-fly with nnAudio) → k-means clustering | pitch, harmony, tonality |

The dual-teacher design is why different depth levels specialize: **low layers suit acoustic tasks** (pitch detection, beat tracking), **high layers suit semantic tasks** (genre, emotion, tagging).

| | MERT-v1-95M | MERT-v1-330M |
|---|---|---|
| Transformer layers | 12 | 24 |
| Hidden size | 768 | 1024 |
| Attention heads | 12 | 16 |
| FFN dim | 3072 | 4096 |
| Parameters | ~95M | ~330M |
| Hidden states out | 13 | 25 |
| Pre-training data | ~160k music tracks (~10k hours) | ~21M tracks (~160k hours) |

(MERT-v0-public is the reproducible variant trained on public music4all data with a HuBERT-style MFCC k-means acoustic teacher.)

### Figure: the same pipeline, drawn programmatically

The diagram below is generated with matplotlib so it always matches the model actually loaded in this notebook.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 7))
ax.axis('off')
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)

def box(x, y, w, h, text, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.12',
                                fc=color, ec='none'))
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=9.5)

def arrow(x, y1, y2):
    ax.annotate('', xy=(x, y2), xytext=(x, y1),
                arrowprops=dict(arrowstyle='-|>', lw=1.6, color='0.35'))

box(2.5, 8.8, 5, 0.9, 'waveform  (1, T) @ 24 kHz', '#cfe8ff')
arrow(5, 8.8, 8.35)
box(1.5, 6.9, 7, 1.45, 'Conv Feature Encoder\n7 × Conv1d · 512 ch\nstrides 5,2,2,2,2,2,2 → ÷320 (75 fps)', '#bfe3c8')
arrow(5, 6.9, 6.45)
box(2.5, 5.5, 5, 0.95, 'Feature Projection\nLayerNorm + Linear 512 → 768', '#ffe3b3')
arrow(5, 5.5, 5.05)
box(1.5, 3.3, 7, 1.75, 'Transformer Encoder × 12\nLN → MHSA (12 heads) → LN → FFN 768→3072', '#e6d4f5')
arrow(5, 3.3, 2.85)
box(2.5, 1.9, 5, 0.95, '13 hidden states × (frames, 768)', '#eeeeee')

ax.text(8.8, 4.3, 'low layers → pitch / beat', fontsize=9, color='seagreen')
ax.text(8.8, 3.7, 'high layers → genre / emotion', fontsize=9, color='tomato')
ax.set_title('MERT-v1-95M architecture')
plt.show()

## 2. Environment Check

In [ ]:
import torch

print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    !nvidia-smi -L

In [ ]:
# transformers / librosa ship with Colab; make sure they are recent
!pip install -q -U transformers librosa

## 3. Load MERT

Weights are ~380 MB and download automatically from Hugging Face on first run (Colab servers reach HF directly — no mirror needed). Note that MERT ships its own modeling code on the Hub (`modeling_MERT.py`), so loading requires `trust_remote_code=True`.

In [ ]:
from transformers import AutoModel, Wav2Vec2FeatureExtractor

MODEL_ID = 'm-a-p/MERT-v1-95M'   # swap for 'm-a-p/MERT-v1-330M' if you have spare VRAM (~1.3 GB)
SR = 24000                       # MERT input sample rate
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
# trust_remote_code=True: MERT ships its own modeling_MERT.py on the Hub
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'model loaded on {device}, {n_params/1e6:.1f}M parameters')

## 4. Anatomy I — Convolutional Feature Encoder

7 stacked 1-D convolutions turn the raw waveform into frame-level features. Only the strides matter for output length: the product of all strides is the **total downsampling factor** (320× → 75 fps).

In [ ]:
# Walk the conv front-end layer by layer
conv_layers = model.feature_extractor.conv_layers
total_stride = 1
for i, layer in enumerate(conv_layers):
    c = layer.conv
    total_stride *= c.stride[0]
    print(f'Conv{i}: {c.in_channels:>3d} -> {c.out_channels:>3d} ch | '
          f'kernel {c.kernel_size[0]:>2d} | stride {c.stride[0]}')

print(f'\ntotal downsample factor : {total_stride}x')
print(f'frame rate              : {SR} / {total_stride} = {SR/total_stride:.0f} fps')

## 5. Anatomy II — Feature Projection & Transformer Encoder

The 512-dim conv output is projected to the model dimension, then passes through 12 Transformer layers. `output_hidden_states=True` gives you every layer: **index 0 is the projected conv output, indices 1–12 are the Transformer layers**.

In [ ]:
# Feature projection: LayerNorm + Linear(512 -> 768)
print(model.feature_projection)

# Transformer encoder configuration
cfg = model.config
print(f'\nhidden_size         : {cfg.hidden_size}')
print(f'num_hidden_layers   : {cfg.num_hidden_layers}')
print(f'num_attention_heads : {cfg.num_attention_heads}')
print(f'intermediate_size   : {cfg.intermediate_size}   (FFN 768 -> 3072 -> 768)')
print(f'hidden_states out   : 1 + {cfg.num_hidden_layers} = {1 + cfg.num_hidden_layers}')

# Uncomment to print the full module tree:
# print(model)

## 6. Prepare Audio (24 kHz mono)

MERT expects **24 kHz mono** input. Below we use two built-in librosa example tracks (auto-downloaded): an orchestral piece and a solo trumpet — two very different timbres, which will make the layer-wise comparison interesting. You can also upload your own mp3/wav via the 📁 file panel and pass the path to `load_audio()`.

In [ ]:
import librosa
import numpy as np

MAX_SEC = 60   # cap each clip at 60 s to keep VRAM in check

def load_audio(path, sr=SR, max_sec=MAX_SEC):
    wav, _ = librosa.load(path, sr=sr, mono=True)
    return wav[: sr * max_sec]

audio_a = load_audio(librosa.example('nutcracker'))   # orchestral (Tchaikovsky)
audio_b = load_audio(librosa.example('trumpet'))     # solo trumpet

print(f'audio A (orchestral): {len(audio_a)/SR:.1f}s')
print(f'audio B (trumpet)   : {len(audio_b)/SR:.1f}s')

### Figure: waveforms

The time-domain view: orchestral music is dense and continuous, the trumpet recording is sparser with clear note onsets. This is exactly the signal the 7-layer conv encoder will turn into 75 fps frame features.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5))
for ax, wav, title, color in [(axes[0], audio_a, 'Audio A — orchestral (nutcracker)', 'tab:blue'),
                              (axes[1], audio_b, 'Audio B — solo trumpet', 'tab:orange')]:
    t = np.arange(len(wav)) / SR
    ax.plot(t[::50], wav[::50], lw=0.4, color=color)   # decimated for speed
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('amplitude')
plt.tight_layout()
plt.show()

In [ ]:
# "Always take a moment to listen to the data" — ISMIR music classification tutorial.
from IPython.display import Audio, display
display(Audio(audio_a, rate=SR))   # A — orchestral (nutcracker)
display(Audio(audio_b, rate=SR))   # B — solo trumpet

### Figure: log-mel spectrogram

The time–frequency view of audio A. Horizontal bands are sustained harmonics, vertical stripes are percussive onsets. Note the connection to MERT's pre-training: its **music teacher** operates on CQT (constant-Q transform), a log-frequency representation closely related to what you see here — that is how pitch and harmony information gets baked into the model.

In [ ]:
import librosa.display

S = librosa.feature.melspectrogram(y=audio_a, sr=SR, n_mels=128)
S_dB = librosa.power_to_db(S, ref=np.max)

plt.figure(figsize=(10, 4))
librosa.display.specshow(S_dB, sr=SR, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title('Audio A — log-mel spectrogram')
plt.tight_layout()
plt.show()

## 7. Shape Trace Through Every Stage

One forward pass, printing the tensor shape at each stage — the whole model in one picture. Watch the sample axis shrink by 320× in the conv front-end and then stay fixed through the Transformer stack. Two HuBERT-family quirks to note: the conv encoder returns **channel-first** `(1, 512, frames)`, and the custom `MERTFeatureProjection` returns a **single tensor** (not a tuple).

In [ ]:
inputs = processor(audio_a, sampling_rate=SR, return_tensors='pt').to(device)

with torch.no_grad():
    x = inputs.input_values                                  # (1, samples)
    cnn_out = model.feature_extractor(x)                     # HuBERT-style conv front-end -> (1, 512, frames)
    proj_out = model.feature_projection(cnn_out.transpose(1, 2))  # LN + Linear, returns a single tensor
    outputs = model(**inputs, output_hidden_states=True)
    hs = torch.stack(outputs.hidden_states)                  # (13, frames, 768)

print(f'waveform                  : {tuple(x.shape)}        # 24 kHz samples')
print(f'after conv encoder        : {tuple(cnn_out.shape)}    # (512, frames) channel-first @ 75 fps')
print(f'after feature projection  : {tuple(proj_out.shape)}     # (frames, 768) = hidden state 0')
print(f'all hidden states         : {tuple(hs.shape)}         # 1 CNN + 12 Transformer layers')

## 8. Extract All-Layer Embeddings

In [ ]:
@torch.no_grad()
def extract_embeddings(wav):
    '''Return all hidden states as one tensor of shape (n_layers, frames, 768).'''
    inputs = processor(wav, sampling_rate=SR, return_tensors='pt').to(device)
    outputs = model(**inputs, output_hidden_states=True)
    return torch.stack(outputs.hidden_states).squeeze(1)

emb_a = extract_embeddings(audio_a)
emb_b = extract_embeddings(audio_b)

print(f'embeddings A: {tuple(emb_a.shape)}  (n_layers, frames, dim)')
print(f'embeddings B: {tuple(emb_b.shape)}')

## 9. Layer-Wise Similarity Between Two Tracks

Mean-pool each layer into one track vector, then compute the A/B cosine similarity **per layer**. Expectation from the paper: low layers stay similar (both are acoustic music), while deeper semantic layers separate the two instruments/genres more sharply.

In [ ]:
import torch.nn.functional as F

sims = [F.cosine_similarity(emb_a[i].mean(0), emb_b[i].mean(0), dim=0).item()
        for i in range(emb_a.shape[0])]

for i, s in enumerate(sims):
    print(f'layer {i:2d}: {s:+.4f}  ' + '█' * int(max(s, 0) * 40))

print(f'\nhighest similarity at layer {int(np.argmax(sims))}: {max(sims):.4f}')

### Figure: layer-wise similarity as a chart

Blue = front-end/low Transformer layers (acoustic), red = high Transformer layers (semantic). The dip toward deeper layers is the layer specialization described in the paper — semantic space tells orchestral and trumpet apart much more strongly than raw acoustics does.

In [ ]:
n_layers = len(sims)
colors = ['tab:blue' if i <= 4 else 'tab:red' for i in range(n_layers)]

plt.figure(figsize=(8, 3.5))
plt.bar(range(n_layers), sims, color=colors)
plt.axhline(0, color='gray', lw=0.6)
plt.xlabel('layer index  (0 = conv projection, 1–12 = transformer)')
plt.ylabel('cosine similarity')
plt.title('Layer-wise similarity: orchestral vs trumpet')
plt.tight_layout()
plt.show()

## 10. Self-Similarity Heatmap (Music Structure)

Take a deep layer (12), split the track into 5-second chunks, mean-pool each chunk, and compute pairwise cosine similarity. The diagonal is always bright (a chunk equals itself); off-diagonal bright blocks ≈ recurring themes/sections — a minimal example of structure analysis with MERT.

In [ ]:
def self_similarity(emb, layer=12, sec=5):
    e = emb[layer]                       # (frames, 768)
    n = e.shape[0] // (sec * 75)         # 75 frames per second
    chunks = e[: n * sec * 75].reshape(n, sec * 75, e.shape[-1]).mean(1)
    chunks = F.normalize(chunks, dim=1)
    return chunks @ chunks.T

mat = self_similarity(emb_a).cpu().numpy()
n = len(mat)

plt.figure(figsize=(6, 5))
plt.imshow(mat, cmap='magma', vmin=0, vmax=1)
plt.colorbar(label='cosine similarity')
ticks = np.arange(n) * 5
plt.xticks(range(n), ticks)
plt.yticks(range(n), ticks)
plt.xlabel('Time (s)')
plt.ylabel('Time (s)')
plt.title('Audio A self-similarity (layer 12)')
plt.show()

## 11. Pattern for Downstream Tasks (genre / emotion / tagging)

Freeze MERT, extract one layer (or a learnable weighted sum of layers — paper §3.3), mean-pool, and train a lightweight classifier head on top. Rule of thumb for layer choice: pitch/beat tasks → low layers; genre/emotion → high layers; when in doubt, validate a few layers on a dev split.

In [ ]:
# ===== Skeleton: frozen MERT features + linear classifier =====
# feats, labels = [], []
# for path, label in my_music_dataset:             # your own (audio, label) pairs
#     wav = load_audio(path)
#     feat = extract_embeddings(wav)[12].mean(0).cpu().numpy()   # deep-layer track vector
#     feats.append(feat); labels.append(label)
#
# from sklearn.linear_model import LogisticRegression
# clf = LogisticRegression(max_iter=1000).fit(feats, labels)
#
# Scale up: learnable layer weighting, MERT-v1-330M, or full fine-tuning.

print('Next: section 12 runs exactly this pattern on real data (FMA-small).')

## 12. Real-Data Experiment — FMA Genre Classification, Layer by Layer

Everything so far used two built-in toy tracks. Let's do it for real: download the **FMA-small** dataset (8,000 × 30 s mp3 clips, 8 balanced genres, Creative-Commons licensed), freeze MERT, and train a tiny linear classifier **separately on each of the 13 hidden states** — the classic *linear probing* protocol.

Following the ISMIR tutorial's advice: the encoder stays **frozen**, we train **only a linear head**, and no augmentation at eval. The resulting accuracy-vs-layer curve is the empirical answer to §1's claim: *which depth carries genre semantics?*

> ⏳ Budget: the audio zip is ~7.2 GiB (aria2 parallel download, typically 5–15 min on Colab) and extraction needs ~15 GiB free disk — fine on a standard runtime. **This section is self-contained: skipping it breaks nothing above.** GPU recommended for the feature pass (~3–5 min on a T4).

In [ ]:
# FMA-small: 8,000 x 30-s mp3 clips, 8 balanced genres, Creative-Commons licensed
# aria2 opens parallel connections (the host throttles single connections);
# 7z is FMA's recommended unzipper — plain `unzip` often fails on their zips.
!apt-get -qq install -y aria2 p7zip-full > /dev/null
!aria2c -x16 -s16 -q -o fma_small.zip     https://os.unil.cloud-switch.ch/fma/fma_small.zip
!aria2c -x8        -q -o fma_metadata.zip https://os.unil.cloud-switch.ch/fma/fma_metadata.zip
!7z x -y -bso0 fma_small.zip                                  # -> fma_small/000002.mp3 ...
!7z x -y -bso0 fma_metadata.zip fma_metadata/tracks.csv       # we only need tracks.csv
import os
os.remove('fma_small.zip'); os.remove('fma_metadata.zip')      # reclaim ~7.5 GiB
print('audio clips :', len(os.listdir('fma_small')))
print('metadata    :', os.path.getsize('fma_metadata/tracks.csv') // 1024, 'KiB')

In [ ]:
import pandas as pd

# FMA metadata is a two-level CSV header; track id is the index
tracks = pd.read_csv('fma_metadata/tracks.csv', index_col=0, header=[0, 1])
small = tracks[('set', 'subset')] == 'small'
split = tracks.loc[small, ('set', 'split')]          # FMA ships official train/val/test splits
genre = tracks.loc[small, ('track', 'genre_top')].dropna()

N_TRAIN, N_TEST = 40, 12                             # per genre -> 320 train / 96 test tracks
rng = np.random.default_rng(0)
train_ids, test_ids, labels = [], [], {}
for g in sorted(genre.unique()):
    ids = genre[(genre == g) & (split == 'train')].index.to_numpy()
    tst = genre[(genre == g) & (split == 'test')].index.to_numpy()
    for i in rng.choice(ids, N_TRAIN, replace=False):
        labels[i] = g; train_ids.append(i)
    for i in rng.choice(tst, N_TEST, replace=False):
        labels[i] = g; test_ids.append(i)

print(f'{len(train_ids)} train / {len(test_ids)} test tracks across {len(set(labels.values()))} genres:')
print(sorted(set(labels.values())))

In [ ]:
import time

all_ids = train_ids + test_ids
vectors = np.zeros((len(all_ids), 13, 768), dtype=np.float32)   # per-track mean-pooled layer vectors

t0 = time.time()
for k, tid in enumerate(all_ids):
    wav = load_audio(f'fma_small/{tid:06d}.mp3', max_sec=30)
    vectors[k] = extract_embeddings(wav).mean(dim=1).cpu().numpy()
    if (k + 1) % 64 == 0:
        print(f'  {k+1}/{len(all_ids)} tracks done ({time.time()-t0:.0f}s)')

np.save('fma_mert_features.npy', vectors)   # cached: re-run probes without the GPU pass
print(f'features: {vectors.shape} (tracks, layers, dim) in {time.time()-t0:.0f}s')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

n_train = len(train_ids)
y_train = [labels[i] for i in train_ids]
y_test  = [labels[i] for i in test_ids]

accs = []
for layer in range(13):
    probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    probe.fit(vectors[:n_train, layer], y_train)      # train: linear head only
    accs.append(probe.score(vectors[n_train:, layer], y_test))
    tag = 'conv' if layer == 0 else 'tfm'
    print(f'layer {layer:2d} ({tag}): test accuracy {accs[-1]:.3f}')

print(f'\nchance level 1/8 = 0.125 | best layer = {int(np.argmax(accs))} '
      f'({max(accs):.3f})')

In [ ]:
best = int(np.argmax(accs))

plt.figure(figsize=(8, 4))
plt.plot(range(13), accs, color='tab:blue', lw=2, marker='o', ms=6, zorder=3)
plt.axhline(1/8, color='0.55', ls='--', lw=1.2, zorder=2)          # chance-level reference
plt.text(12.4, 1/8 + 0.012, 'chance (1/8)', ha='right', fontsize=9, color='0.4')
plt.annotate(f'best: layer {best}\n{accs[best]:.2f}', xy=(best, accs[best]),
             xytext=(best, accs[best] + 0.09), ha='center', fontsize=9,
             arrowprops=dict(arrowstyle='-', color='0.4', lw=0.8))
plt.xticks(range(13))
plt.ylim(0, max(accs) * 1.25)
plt.xlabel('hidden-state layer  (0 = conv projection, 1-12 = transformer)')
plt.ylabel('test accuracy')
plt.title('Linear-probe genre accuracy by MERT layer (FMA-small)')
plt.grid(axis='y', color='0.92', zorder=0)
plt.tight_layout()
plt.show()

### Reading the curve

- Every layer beats the 12.5% chance level by a wide margin — even the raw conv projection (layer 0) carries genre signal, because a 30-second mean-pooled spectrum already separates broad styles.
- Accuracy typically climbs through the middle layers and peaks in the upper Transformer stack — genre is a *semantic* property, exactly the regime where the paper says deep MERT layers specialize. Compare with §9, where the deep layers were also the ones telling orchestral and trumpet apart.
- **Takeaway for practice:** don't hardcode the last layer. The MERT paper's downstream recipe (and MERBench) uses a *learnable weighted sum of all 13 hidden states*, so the data picks the depth. The probe above is the cheap diagnostic that tells you whether that extra machinery is worth it for your task.

Easy follow-ups: raise `N_TRAIN`/`N_TEST` for less noisy curves, or point `MODEL_ID` at MERT-v1-330M and see whether deeper stacks shift the peak.

## References & Further Reading

**MERT**
- Paper: [MERT: Acoustic Music Understanding Model with Large-scale Self-supervised Training](https://arxiv.org/abs/2306.00107) (ICLR 2024)
- Code: <https://github.com/yizhilll/MERT> · Models: [MERT-v1-95M](https://huggingface.co/m-a-p/MERT-v1-95M) · [MERT-v1-330M](https://huggingface.co/m-a-p/MERT-v1-330M)

**The self-supervised landscape MERT belongs to**
- Contrastive: [CPC](https://arxiv.org/abs/1807.03748) · [SimCLR](https://arxiv.org/abs/2002.05709) · [CLMR](https://arxiv.org/abs/2103.09410) (music version of SimCLR)
- Masked prediction: [HuBERT](https://arxiv.org/abs/2106.07447) → **MERT** (this notebook) → follow-ups [MULE](https://arxiv.org/abs/2210.03799), [MusicFM](https://arxiv.org/abs/2311.03318)
- Audio–text joint embeddings: [MuLan](https://arxiv.org/abs/2208.04473), CLAP

**Data**
- FMA dataset: <https://github.com/mdeff/fma> (Defferrard et al., ISMIR 2017) — used in §12

**Courses & tutorials (recommended)**
- KAIST GCT634, *Musical Applications of Machine Learning* — [Music Representation Learning lecture](https://mac.kaist.ac.kr/~juhan/gct634/) (Juhan Nam)
- ISMIR tutorial — [Music Classification: Beyond Supervised Learning · Self-Supervised Learning](https://music-classification.github.io/tutorial/part5_beyond/self-supervised-learning.html) (the linear-probing protocol in §12 follows its advice)